# Calculate outcomes and admissions by LSOA

Outcomes assume that the whole population is treated.

__Results__:

For the base scenario and with adding MT to both Taunton and Worcester, and for including and excluding lossy drip-and-ship transfers, save these files:
+ `results_full_{scenario_name}.csv`
+ `results_stats_{scenario_name}.csv`
+ `results_stats_by_unit_{scenario_name}.csv`


## Code setup

In [1]:
import os
import numpy as np
import pandas as pd

from classes.add_net_benefit import add_net_benefit, add_mt_and_transfers, \
    add_admissions_weighted_outcomes

In [2]:
dir_output = 'output'
file_geo_outcomes = 'results_geo_outcomes_'  # + scenario.csv

## Parameters

There must be a matching geo_outcomes file for each of these scenarios:

In [6]:
scenarios = ['base', 'taunworc']

Fixed proportion of patients have LVO (rest have nLVO), and options for proportion transferred for MT are gathered in the same results file.

In [4]:
prop_lvo = 0.3
options_prop_lvo_dripship_treated_with_mt = [1.0, 0.6]

# MT rate only affects admissions and transfers, not outcomes:
mt_rate = 0.1

col_admissions = 'Admissions_2025_Est'

## Run for all scenarios: main changes

Save a copy of the full results to file.

For each set of changed units, calculate the new geography and outcomes data. Then for each set of stroke population parameters, combine the outcomes into the full population.

In [7]:
for scenario in scenarios:
    # Load data:
    file_geo_outcomes_here = file_geo_outcomes + f'{scenario}.csv'
    p = os.path.join(dir_output, file_geo_outcomes_here)
    df_geo_outcomes = pd.read_csv(p, index_col='LSOA')
    
    results = df_geo_outcomes.copy()
    cols_start = df_geo_outcomes.columns
    list_of_dfs = [results]
    for p in options_prop_lvo_dripship_treated_with_mt:
        # Pick out info for this scenario:
        s_p = '' if p == 1.0 else f'_transfer{p}'.replace('.', '!')
        scen_name = f'{scenario}{s_p}'
    
        # Don't overwrite shared results:
        results_here = df_geo_outcomes.copy()
        
        # Calculate numbers of thrombectomies and transfers:
        results_here = add_mt_and_transfers(
            results_here, col_admissions, mt_rate, prop_lvo_ds_mt=p)
        # Calculate outcome results that depend on transfers:
        # Add net benefit columns
        results_here = add_net_benefit(
            results_here,
            prop_lvo=prop_lvo,
            prop_lvo_ds_mt=p
        )
        # Calculate admissions-weighted outcomes:
        cols_outcome = ['drip_ship_mrs_0-2', 'lvo_drip_ship_ivt_mt_mrs_0-2']
        if p != 1.0:
            cols_outcome = [f'{c}_transferloss' for c in cols_outcome]
        results_here = add_admissions_weighted_outcomes(
            results_here, col_admissions, cols_outcome)
    
        old_cols = sum([df.columns.tolist() for df in list_of_dfs], [])
        new_cols = [c for c in results_here.columns if c not in old_cols]
        if p != 1.0:
            rename_dict = dict([(c, f'{c}{p}'.replace('.', '!'))
                for c in new_cols])
        else:
            rename_dict = dict([(c, c) for c in new_cols])
        list_of_dfs.append(results_here[new_cols].rename(columns=rename_dict))
    
    results = pd.concat(list_of_dfs, axis='columns')
    cols_new = [c for c in results.columns if c not in cols_start]
    
    # Round outcome results:
    cols_mrs = [c for c in results.columns if (
                ('mrs' in c) | ('improvement' in c))]
    cols_util = [c for c in results.columns if ('util' in c)]
    results[cols_mrs] = results[cols_mrs].round(3)
    results[cols_util] = results[cols_util].round(5)
    
    # Get descriptive stats for all numeric columns
    descriptive_stats = results.describe(percentiles=[0.25, 0.5, 0.75]).T
    descriptive_stats = descriptive_stats.round(5)
    
    # Repeat descriptive stats, grouped by nearest_ivt_unit
    results_stats_by_unit = results.groupby('nearest_ivt_unit').describe().T
    # Unpack index column so two-level index becomes two columns
    results_stats_by_unit = results_stats_by_unit.reset_index()
    rename_dict = {'level_0': 'variable', 'level_1': 'statistic'}
    results_stats_by_unit = results_stats_by_unit.rename(columns=rename_dict)
    results_stats_by_unit = results_stats_by_unit.round(5)
    
    # Save results:
    f = f'results_full_{scenario}.csv'
    results[cols_new].to_csv(os.path.join(dir_output, f))
    
    f = f'results_stats_{scenario}.csv'
    descriptive_stats.to_csv(os.path.join(dir_output, f))
    
    f = f'results_stats_by_unit_{scenario}.csv'
    results_stats_by_unit.to_csv(os.path.join(dir_output, f))

## Conclusion

Calculated full results, descriptive stats, and stats by unit and saved to file.